In [1]:
import sys

!{sys.executable} -m pip install kagglehub

In [2]:
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download(
    "asaniczka/uk-optimal-product-price-prediction"
)

print("Dataset folder:", path)
print(os.listdir(path))

Dataset folder: /Users/andreazaffe/.cache/kagglehub/datasets/asaniczka/uk-optimal-product-price-prediction/versions/3
['amz_uk_price_prediction_dataset.csv']


In [3]:
file_path = os.path.join(
    path,
    "amz_uk_price_prediction_dataset.csv"
)

df = pd.read_csv(file_path)

display(df.head())
print(df.shape)
print(df.columns.tolist())

,uid,asin,title,stars,reviews,price,isBestSeller,boughtInLastMonth,category
0,1,B09B96TG33,"Echo Dot (5th generation, 2022 release) | Big ...",4.7,15308,21.99,False,0,Hi-Fi Speakers
1,2,B01HTH3C8S,"Anker Soundcore mini, Super-Portable Bluetooth...",4.7,98099,23.99,True,0,Hi-Fi Speakers
2,3,B09B8YWXDF,"Echo Dot (5th generation, 2022 release) | Big ...",4.7,15308,21.99,False,0,Hi-Fi Speakers
3,4,B09B8T5VGV,"Echo Dot with clock (5th generation, 2022 rele...",4.7,7205,31.99,False,0,Hi-Fi Speakers
4,5,B09WX6QD65,Introducing Echo Pop | Full sound compact Wi-F...,4.6,1881,17.99,False,0,Hi-Fi Speakers


(2443651, 9)
['uid', 'asin', 'title', 'stars', 'reviews', 'price', 'isBestSeller', 'boughtInLastMonth', 'category']


## Part 1: Best-Seller Trends Across Product Categories

We compare product categories with best-seller status to see whether some categories have a higher proportion of best-selling products.

In [5]:
# Create a crosstab showing the number of best-sellers
# and non-best-sellers in each category
best_seller_crosstab = pd.crosstab(
    df["category"],
    df["isBestSeller"]
)

# Rename the boolean columns to clearer text labels
best_seller_crosstab = best_seller_crosstab.rename(
    columns={
        False: "not_best_seller",
        True: "best_seller"
    }
)

# Calculate the total number of products in each category
best_seller_crosstab["total_products"] = (
    best_seller_crosstab["not_best_seller"]
    + best_seller_crosstab["best_seller"]
)

# Calculate the proportion of best-sellers per category
best_seller_crosstab["best_seller_proportion"] = (
    best_seller_crosstab["best_seller"]
    / best_seller_crosstab["total_products"]
)

# Sort from highest to lowest proportion
best_seller_crosstab = best_seller_crosstab.sort_values(
    by="best_seller_proportion",
    ascending=False
)

# Show the top 10 categories
best_seller_crosstab.head(10)

isBestSeller,not_best_seller,best_seller,total_products,best_seller_proportion
category,,,,
Grocery,9008,556,9564,0.058135
Smart Home Security & Lighting,98,6,104,0.057692
Health & Personal Care,9017,552,9569,0.057686
Mobile Phone Accessories,248,11,259,0.042471
Power & Hand Tools,8353,306,8659,0.035339
"Billiard, Snooker & Pool",241,8,249,0.032129
Pet Supplies,9152,285,9437,0.030200
Home Brewing & Wine Making,239,7,246,0.028455
Wind Instruments,243,7,250,0.028000


### Chi-square Test and Cramér's V

We test whether best-seller status is related to product category.  
The chi-square test checks whether the two variables are independent. Cramér's V measures how strong the relationship is.

In [6]:
from scipy.stats import chi2_contingency
import numpy as np

# Create a clean contingency table with counts only
contingency_table = pd.crosstab(
    df["category"],
    df["isBestSeller"]
)

# Run the chi-square test
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

# Calculate Cramér's V
n = contingency_table.to_numpy().sum()
rows, columns = contingency_table.shape

cramers_v = np.sqrt(
    chi2 / (n * min(rows - 1, columns - 1))
)

print("Chi-square statistic:", chi2)
print("P-value:", p_value)
print("Degrees of freedom:", dof)
print("Cramér's V:", cramers_v)

Chi-square statistic: 36540.20270061387
P-value: 0.0
Degrees of freedom: 295
Cramér's V: 0.1222829439760564
